In [1]:
from pyspark.sql import SparkSession

# Création de la session Spark
# .master("local[*]") signifie qu'on utilise tous les cœurs du CPU de la machine
spark = SparkSession.builder \
    .appName("TP_RDD_Operations") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
print("Spark est prêt ! Version :", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/19 06:34:23 WARN Utils: Your hostname, codespaces-e1abfc, resolves to a loopback address: 127.0.0.1; using 10.0.3.234 instead (on interface eth0)
26/03/19 06:34:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/19 06:34:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark est prêt ! Version : 4.1.1


In [2]:
# Une liste de prix de voitures (en euros)
prix_data = [15000, 22000, 8500, 45000, 12000, 31000, 9500]

# Transformation de la liste en RDD (Distribution sur le cluster)
rdd_prix = sc.parallelize(prix_data)

print("Type de l'objet :", type(rdd_prix))
# On utilise une Action (collect) pour voir le contenu
print("Contenu du RDD :", rdd_prix.collect())

Type de l'objet : <class 'pyspark.core.rdd.RDD'>
Contenu du RDD : [15000, 22000, 8500, 45000, 12000, 31000, 9500]


In [3]:
print(f'les 2 premiers éléments du RDD sont {rdd_prix.take(2)}') 
print(f"Le premier élément du RDD est : {rdd_prix.first()}")

les 2 premiers éléments du RDD sont [15000, 22000]
Le premier élément du RDD est : 15000


## Prise en main des RDD

Dans cette section, on manipule les RDD avec des exemples simples :
- transformations (`map`, `filter`)
- actions (`count`, `reduce`, `takeOrdered`)
- paires clé/valeur (`reduceByKey`, `sortByKey`)
- persistance (`cache`)

In [5]:
# 1) Transformations de base : map et filter

# On applique une remise de 10% 
rdd_remise = rdd_prix.map(lambda p: int(p * 0.9))
# puis on garde seulement les prix >= 10 000€
rdd_remise_filtree = rdd_remise.filter(lambda p: p >= 10000)
print("Recette enregistrée  dans le DAG de Spark !")



Recette enregistrée  dans le DAG de Spark !


In [6]:
# On déclenche le calcul avec des actions
print("Prix après remise :", rdd_remise.collect())
print("Prix après remise et filtrage :", rdd_remise_filtree.collect())

Prix après remise : [13500, 19800, 7650, 40500, 10800, 27900, 8550]
Prix après remise et filtrage : [13500, 19800, 40500, 10800, 27900]


In [7]:
# 2) Actions de base : count, reduce, takeOrdered

nb_voitures = rdd_prix.count()
prix_total = rdd_prix.reduce(lambda a, b: a + b)
prix_moyen = prix_total / nb_voitures

print("Nombre de voitures :", nb_voitures)
print("Somme des prix :", prix_total)
print("Prix moyen :", round(prix_moyen, 2))
print("Les 3 plus petits prix :", rdd_prix.takeOrdered(3))
print("Les 3 plus grands prix :", rdd_prix.takeOrdered(3, key=lambda x: -x))

Nombre de voitures : 7
Somme des prix : 143000
Prix moyen : 20428.57
Les 3 plus petits prix : [8500, 9500, 12000]
Les 3 plus grands prix : [45000, 31000, 22000]


In [8]:
# 3) RDD clé/valeur : reduceByKey et tri par clé

# On crée un RDD (catégorie, prix)
categories = [
    ("citadine", 8500),
    ("berline", 22000),
    ("SUV", 45000),
    ("citadine", 9500),
    ("berline", 31000),
    ("SUV", 12000),
]

rdd_categorie_prix = sc.parallelize(categories)

# Prix total par catégorie
totaux_par_categorie = rdd_categorie_prix.reduceByKey(lambda a, b: a + b)
print("Totaux par catégorie :", totaux_par_categorie.collect())

# Nombre d'éléments par catégorie
nb_par_categorie = rdd_categorie_prix.mapValues(lambda _: 1).reduceByKey(lambda a, b: a + b)
print("Nombre d'éléments par catégorie :", nb_par_categorie.collect())

print("Trié par catégorie :", totaux_par_categorie.sortByKey().collect())

Totaux par catégorie : [('citadine', 18000), ('berline', 53000), ('SUV', 57000)]
Nombre d'éléments par catégorie : [('citadine', 2), ('berline', 2), ('SUV', 2)]
Trié par catégorie : [('SUV', 57000), ('berline', 53000), ('citadine', 18000)]


In [9]:
# 4) Persistance : cache pour éviter de recalculer

rdd_carres = rdd_prix.map(lambda p: p * p).cache()

# Première action: déclenche le calcul + met en cache
print("Carrés (premiers éléments) :", rdd_carres.take(3))

# Deuxième action: réutilise le résultat déjà en mémoire
print("Nombre de carrés :", rdd_carres.count())

Carrés (premiers éléments) : [225000000, 484000000, 72250000]
Nombre de carrés : 7


## Exercice : Refaire un Word Count avec un fichier

Objectif : reproduire le principe MapReduce pour compter les mots d'un fichier texte avec des RDD.

Consigne :
1. Charger le fichier `/Data/MapReduce.txt` avec `sc.textFile(...)`.
2. Transformer les lignes en mots (`flatMap`).
3. Nettoyer les mots (par exemple passer en minuscules et ignorer la ponctuation simple).
4. Créer des paires `(mot, 1)` avec `map`.
5. Agréger avec `reduceByKey`.
6. Afficher :
   - le nombre total de mots,
   - les 10 mots les plus fréquents.

Bonus : filtrer les mots vides (ou très courts) avant l'agrégation.

In [ ]:
# Squelette de départ (à compléter)
import re

# 1) Charger le fichier texte
# Le fichier du TP est: /Data/MapReduce.txt
rdd_lignes = sc.textFile("/Data/MapReduce.txt")

# 2) Découper en mots
# TODO: remplacer "..." par votre transformation
rdd_mots = rdd_lignes.flatMap(lambda ligne: ...)

# 3) Nettoyer les mots (minuscules + retirer ponctuation simple)
# TODO: compléter le nettoyage
rdd_mots_propres = rdd_mots.map(lambda mot: re.sub(r"[^a-zA-Z0-9]", "", ...).lower())

# 4) Filtrer les mots vides (et optionnellement les mots trop courts)
# TODO: compléter le filtre
rdd_mots_filtres = rdd_mots_propres.filter(lambda mot: ...)

# 5) Créer les paires (mot, 1)
# TODO: compléter le map
rdd_paires = rdd_mots_filtres.map(lambda mot: ...)

# 6) Agréger les comptes par mot
# TODO: compléter le reduceByKey
rdd_comptes = rdd_paires.reduceByKey(lambda a, b: ...)

# 7) Afficher les résultats demandés
# TODO: calculer le nombre total de mots
total_mots = ...

# TODO: récupérer le top 10 (mot, fréquence), tri décroissant par fréquence
top10 = ...

print("Nombre total de mots :", total_mots)
print("Top 10 des mots les plus fréquents :", top10)